In [94]:
from collections import namedtuple, defaultdict

import numpy as np
import pandas as pd

from sdrsdm import TriadicMemory

# Setup

In [193]:
RNG = np.random.default_rng(100)
N = 500
P = 5

In [22]:
def my_empty_sdr(n=N, p=P):
    return np.array([], dtype=int)

def my_random_sdr(n=N, p=P):
    inds = RNG.choice(n, p, replace=False)
    inds.sort()
    return inds

def my_sdr_union(s1, s2):
    united = np.r_[s1, s2]
    return np.unique(united)

def my_sdr_overlap(s1, s2):
    s1 = set(s1)
    s2 = set(s2)
    return len(s1 & s2)

def my_sdr_is_equal(s1, s2):
    if len(s1) != len(s2):
        return False

    return np.all(s1 == s2)

In [11]:
codec_cache = namedtuple('CodecCache', ['str_to_sdr', 'sdr_to_str'])({}, {})

def codec(x):
    if x is None:
        return my_empty_sdr()

    if type(x) == np.ndarray:
        if len(x) == 0:
            return None
            
        key = tuple(x)
    
        if key in codec_cache.sdr_to_str:
            return codec_cache.sdr_to_str[key]
        else:
            return x
    elif type(x) == str:
        if x in codec_cache.str_to_sdr:
            return codec_cache.str_to_sdr[x]
        else:
            new_sdr = my_random_sdr()
            codec_cache.str_to_sdr[x] = new_sdr
            codec_cache.sdr_to_str[tuple(new_sdr)] = x
            return new_sdr
    else:
        assert False, f'Unknown type "{type(x)}"'

In [14]:
strs = ['a', 'b', 'c', 'd', 'e']
sdrs = list(map(codec, strs))

for sdr, s in zip(sdrs, strs):
    assert codec(sdr) == s

r = my_random_sdr()
assert np.all(codec(r) == r)
assert len(codec(None)) == 0
assert codec(np.array([])) is None

# How contexts are resolved?

In [134]:
M1 = TriadicMemory(N, P)
M2 = TriadicMemory(N, P)
y, c, u, v, prediction = (my_empty_sdr(),) * 5
clist = []
cset = set()

test_input = 'ABCABC'
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    x = my_sdr_union(y, c)
    y = codec(inp)
    
    if not my_sdr_is_equal(prediction, y):
        M2.store(u, v, y)
    
    c = M1.query_Z(x, y)
    x_hat = M1.query_X(y, c)
    
    if my_sdr_overlap(x_hat, x) < P:
        c = my_random_sdr()
        M1.store(x, y, c)
    
    u = x
    v = y
    prediction = M2.query_Z(u, v)
    
    if not tuple(c) in cset:
        cset.add(tuple(c))
        clist.append(c)

    columns['history'].append(history)
    columns['input'].append(inp)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
    history = history + inp

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,A,None,B,0
1,A,B,None,C,0
2,AB,C,None,A,0
3,ABC,A,None,B,0
4,ABCA,B,C,C,1
5,ABCAB,C,A,$,0


In [140]:
M1.query_Z(codec('A'), codec('B')), clist[1]

(array([194, 260, 291, 384, 464]), array([194, 260, 291, 384, 464]))

In [141]:
M1.query_Z(clist[0], codec('B')), clist[1]

(array([194, 260, 291, 384, 464]), array([194, 260, 291, 384, 464]))

В M1 контексты сохраняются с ключами: x = prev_context | prev_input, y = input. А раз так, то контекст можно разрезловить по частичному совпадению ключей, например:
1) x = prev_input, y = input
2) x = prev_context, y = input   

In [135]:
x1 = my_sdr_union(codec('A'), my_sdr_union(my_random_sdr(), my_random_sdr()))
y1 = my_sdr_union(codec('B'), my_sdr_union(my_random_sdr(), my_random_sdr()))
M1.query_Z(x1, y1), clist[1]

(array([194, 260, 291, 384, 464]), array([194, 260, 291, 384, 464]))

Видно, что добавление лишнего шума (по два СДР для x1 и y1), всё равно позволяют корректно разрешить контекст 'AB'.

In [142]:
codec(M2.query_Z(codec('A'), codec('B')))

'C'

Даже без указания контекста M2 для AB возвращает C

In [144]:
# добавим в M2 Z для AB, т . е . AB -> Z
M2.store(codec('A'), codec('B'), codec('Z'))
codec(M2.query_Z(codec('A'), codec('B')))

array([ 35, 114, 205, 237, 261, 323, 364, 367, 407, 449])

после добавления Z M2 будет возвращать суперпозицию C и Z

In [146]:
codec(M2.query_Z(my_sdr_union(codec('A'), clist[0]), codec('B')))

'C'

но, если указать контекст clist[10, то M2 будет возвращать C. То есть без дополнительной подсказки возвращается всё, что знаем (суперпозиция), с подсказкой - уже конкретное

# Temporal memory

In [176]:
def create_context_mem():
    mem = TriadicMemory(N, P)
    prev_inp, context = (my_empty_sdr(),) * 2

    def process(inp):
        nonlocal prev_inp, context
        
        if len(inp) == 0:
            # Flush
            prev_inp, context = (my_empty_sdr(),) * 2
            return context
        
        x = my_sdr_union(prev_inp, context)
        context = mem.query_Z(x, inp)
        x_hat = mem.query_X(inp, context) # cleanup via roundtrip
        
        if my_sdr_overlap(x_hat, x) < P:
            context = my_random_sdr()
            mem.store(x, inp, context)

        prev_inp = inp
        return context

    return process

In [191]:
context_mem = create_context_mem()
mem = TriadicMemory(N, P)
prev_inp, u, v, prediction = (my_empty_sdr(),) * 4

test_input = 'banana' * 5
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    inp_orig = inp
    inp = codec(inp)
    context = context_mem(inp)

    if not my_sdr_is_equal(prediction, inp):
        mem.store(u, v, inp)

    u, v = my_sdr_union(prev_inp, context), inp
    prediction = mem.query_Z(u, v)
    prev_inp = inp
    
    columns['history'].append(history)
    columns['input'].append(inp_orig)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
    history = history + inp_orig

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,b,None,a,0
1,b,a,None,n,0
2,ba,n,None,a,0
3,ban,a,None,n,0
4,bana,n,a,a,1
5,banan,a,n,b,0
6,banana,b,n,a,0
7,bananab,a,n,n,1
8,bananaba,n,a,a,1
9,bananaban,a,"[51, 183, 205, 259, 297, 343, 395, 470, 482, 494]",n,0


Результаты (по колонке is_correct_pred) соответствуют тем, что в ноутбукe "Temporal Memory Elementary Algorithm.nb"

![](./img/temporal-mathematica-banana.jpg)

# Deep temporal memory

In [202]:
# R1 = create_context_mem()
# R2 = create_context_mem()
# R3 = create_context_mem()
# R4 = create_context_mem()
# R5 = create_context_mem()
# R6 = create_context_mem()
# R7 = create_context_mem()
R_list = list(map(lambda _: create_context_mem(), range(7)))
mem = TriadicMemory(N, P)
prev_inp, context, x, y, prediction = (my_empty_sdr(),) * 5

test_input = 'banana' * 5
history = ''
columns = defaultdict(list)

for inp, next_inp in zip(test_input, test_input[1:] + '$'):
    inp_hat = codec(inp)
    # t1 = R1(inp_hat)
    # t2 = R2(t1)
    # t3 = R3(t2)
    # t4 = R4(t3)
    # t5 = R5(t4)
    # t6 = R6(t5)
    # t7 = R7(t6)
    t_list = []
    
    for R in R_list:
        t = t_list[-1] if t_list else inp_hat
        t_list.append(R(t))

    if not my_sdr_is_equal(prediction, inp_hat):
        mem.store(x, y, inp_hat)

    # x = my_sdr_union(t1, t4)
    # y = my_sdr_union(t2, t7)
    x = my_sdr_union(t_list[0], t_list[3])
    y = my_sdr_union(t_list[1], t_list[6])

    
    prediction = mem.query_Z(x, y)

    columns['history'].append(history)
    columns['input'].append(inp)
    columns['next_pred'].append(codec(prediction))
    columns['next_true'].append(next_inp)
    columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
    history = history + inp

pd.DataFrame(columns)

,history,input,next_pred,next_true,is_correct_pred
0,,b,None,a,0
1,b,a,None,n,0
2,ba,n,None,a,0
3,ban,a,None,n,0
4,bana,n,None,a,0
5,banan,a,n,b,0
6,banana,b,None,a,0
7,bananab,a,n,n,1
8,bananaba,n,a,a,1
9,bananaban,a,n,n,1


Результаты (по колонке is_correct_pred) соответствуют тем, что в ноутбукe "Temporal Memory Elementary Algorithm.nb"

![](./img/temporal-mathematica-banana-2.jpg)

# Consume and continue

In [197]:
R_list = list(map(lambda _: create_context_mem(), range(7)))
# R1 = create_context_mem()
# R2 = create_context_mem()
# R3 = create_context_mem()
# R4 = create_context_mem()
# R5 = create_context_mem()
# R6 = create_context_mem()
# R7 = create_context_mem()
# mem = TriadicMemory(N, P)
# prev_inp, context, x, y, prediction = (my_empty_sdr(),) * 5

# test_input = 'banana' * 5
# history = ''
# columns = defaultdict(list)

# for inp, next_inp in zip(test_input, test_input[1:] + '$'):
#     inp_hat = codec(inp)
#     t1 = R1(inp_hat)
#     t2 = R2(t1)
#     t3 = R3(t2)
#     t4 = R4(t3)
#     t5 = R5(t4)
#     t6 = R6(t5)
#     t7 = R7(t6)

#     if not my_sdr_is_equal(prediction, inp_hat):
#         mem.store(x, y, inp_hat)

#     x = my_sdr_union(t1, t4)
#     y = my_sdr_union(t2, t7)
#     prediction = mem.query_Z(x, y)

#     columns['history'].append(history)
#     columns['input'].append(inp)
#     columns['next_pred'].append(codec(prediction))
#     columns['next_true'].append(next_inp)
#     columns['is_correct_pred'].append(int(my_sdr_is_equal(codec(next_inp), prediction)))
#     history = history + inp

# pd.DataFrame(columns)